# Install Libraries

In [ ]:
!pip install hmmlearn 

# Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import yfinance as yf

from hmmlearn.hmm import GaussianHMM

# Download European Market Data

In [ ]:
ticker = "SAP.DE"

data = yf.download(ticker,start="2010-01-01")

df = data[["Close"]]

df.columns=["price"]

df.head()

# Feature Engineering

In [ ]:
df["returns"] = df["price"].pct_change()

df["volatility"] = df["returns"].rolling(20).std()

df = df.dropna()

df.head()

# Prepare HMM Data

In [ ]:
X = df[["returns","volatility"]].values

# Train Hidden Markov Model

In [ ]:
model = GaussianHMM(
    n_components=2,
    covariance_type="full",
    n_iter=1000
)

model.fit(X)

# Predict Market Regimes

In [ ]:
hidden_states = model.predict(X)

df["regime"] = hidden_states

# Visualize Regimes

In [ ]:
plt.figure(figsize=(12,6))

for regime in range(2):
    
    subset = df[df["regime"]==regime]
    
    plt.scatter(
        subset.index,
        subset["price"],
        label=f"Regime {regime}",
        s=10
    )

plt.legend()
plt.title("Market Regimes Detected by Hidden Markov Model")
plt.show()

# Analyze Regime Characteristics

In [ ]:
df.groupby("regime")[["returns","volatility"]].mean()

* **Regime 0 → low volatility bull market** 




* **Regime 1 → high volatility bear market**


# Trading Strategy

Strategy rules:

**If regime = bull → long**




**If regime = bear → cash**

# Strategy Implementation

In [ ]:
df["position"] = 0

bull_regime = df.groupby("regime")["returns"].mean().idxmax()

df.loc[df["regime"]==bull_regime,"position"] = 1

# Strategy Returns

In [ ]:
df["strategy_returns"] = df["position"].shift(1) * df["returns"]

df["cum_strategy"] = (1+df["strategy_returns"]).cumprod()

df["cum_market"] = (1+df["returns"]).cumprod()

# Performance Plot

In [ ]:
returns = df["strategy_returns"].dropna()

sharpe = np.sqrt(252)*returns.mean()/returns.std()

max_drawdown = (df["cum_strategy"]/df["cum_strategy"].cummax()-1).min()

print("Sharpe Ratio:",sharpe)

print("Max Drawdown:",max_drawdown)

# Advanced Visualisation

In [ ]:
plt.figure(figsize=(12,4))

plt.plot(df["regime"])

plt.title("Detected Market Regimes")

plt.show()

# Transition Matrix Analysis

In [ ]:
model.transmat_